# Lab 13 — Book Analytics Live Dashboard

This notebook accompanies **Lab 13**, which extends the Book Analytics Pipeline
into a fully interactive, deployable web application built with **Dash**.

Earlier labs produced *static* charts saved as files — a stakeholder could only
see one fixed view. This lab turns the same cleaned book dataset into a **live
Dash dashboard** that anyone with the URL can filter and explore, served by
**Gunicorn** and orchestrated together with **MongoDB** via **Docker Compose**.

**Dataset note.** The project works with the cleaned **OpenLibrary book**
dataset (`data/processed/cleaned/cleaned_books.csv`) — each record is a book
with a title, genre, publication year, rating, ratings count, page count and
edition count.

### Learning objectives
- Build a Dash app with `html`, `dcc` and `dash-bootstrap-components`.
- Implement reactive callbacks that filter and redraw Plotly charts.
- Connect callbacks to a MongoDB data source (with a graceful CSV fallback).
- Stream simulated live data into a chart with `dcc.Interval`.
- Serve the app with Gunicorn and containerise it with Docker / Docker Compose.

### New files in this lab
```
app.py                       # Dash entry point (creates app, loads layout, registers callbacks)
Dockerfile                   # python:3.12-slim + Gunicorn
docker-compose.yml           # web (dashboard) + db (MongoDB)
.dockerignore                # keeps the image lean
requirements-dashboard.txt   # lean deps for the image
src/dashboard/__init__.py    # package marker
src/dashboard/data_access.py # MongoDB queries + CSV fallback + filter helpers
src/dashboard/layout.py      # page structure (KPI cards, filters, charts, ticker)
src/dashboard/callbacks.py   # interactive @callback functions
scripts/seed_mongo.py        # one-time MongoDB seeding
```

## Part 1 — Install required libraries

Dash runs on Flask and renders with React + Plotly. Install the dashboard
stack (already added to `requirements.txt`):

| Library | Role |
|---|---|
| `dash` | Core framework (Flask + React + Plotly wiring) |
| `dash-bootstrap-components` | Responsive grid, cards, navbars |
| `pymongo` | Python driver for MongoDB |
| `gunicorn` | Production WSGI server (replaces the Flask dev server) |
| `websockets` | Async client for live data streams (reference pattern) |

In [1]:
# Run once if the packages aren't installed yet:
# %pip install dash dash-bootstrap-components pymongo gunicorn websockets

import dash, dash_bootstrap_components as dbc, pymongo, plotly
print("dash:", dash.__version__)
print("dash-bootstrap-components:", dbc.__version__)
print("pymongo:", pymongo.__version__)
print("plotly:", plotly.__version__)

c:\Users\Laptop\Desktop\Unstructured Data Project\Unstructured-Data-Project\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


dash: 4.2.0
dash-bootstrap-components: 2.0.4
pymongo: 4.16.0
plotly: 6.7.0


## Part 2 — Make the `src/` package importable

This notebook lives in `notebooks/`, so we add the project's `src/` folder to
`sys.path` (exactly like `app.py` does) before importing the dashboard
package.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", PROJECT_ROOT)
print("src on path:", str(SRC) in sys.path)

Project root: c:\Users\Laptop\Desktop\Unstructured Data Project\Unstructured-Data-Project
src on path: True


## Part 3 — Seed MongoDB (one time)

`scripts/seed_mongo.py` reads the cleaned CSV, drops the `books` collection,
re-inserts every record and builds indexes on `genres`, `first_publish_year`
and `title`. Run it whenever the database needs refreshing.

> If MongoDB isn't running this cell will report an error — that's fine, the
> dashboard automatically falls back to the cleaned CSV (Part 4).

In [3]:
# Programmatic seeding (equivalent to: python scripts/seed_mongo.py)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
try:
    import seed_mongo
    inserted = seed_mongo.seed()
    print(f"Seeded {inserted} documents into MongoDB.")
except Exception as exc:
    print("Seeding skipped/failed (dashboard will use the CSV fallback):", exc)

2026-06-03 11:41:14,320 - INFO - Read 100 rows from C:\Users\Laptop\Desktop\Unstructured Data Project\Unstructured-Data-Project\data\processed\cleaned\cleaned_books.csv
2026-06-03 11:41:14,377 - INFO - Inserted 100 documents into book_pipeline.books
2026-06-03 11:41:14,498 - INFO - Created indexes on genres, first_publish_year, title


Seeded 100 documents into MongoDB.


## Part 4 — Data access layer (MongoDB → CSV fallback)

`src/dashboard/data_access.py` is the single place that knows how to fetch
data. It tries **MongoDB first**; on any failure (server down, empty
collection) it falls back **gracefully** to the cleaned CSV. The rest of the
app never has to care which source was used.

In [4]:
from dashboard import data_access as da

df = da.load_data()
print("Loaded from:", da.get_source().upper())
print("Shape:", df.shape)
df.head()

2026-06-03 11:41:14,530 - INFO - Loaded 100 documents from MongoDB (book_pipeline.books)
2026-06-03 11:41:14,537 - INFO - Dashboard dataset ready: 100 rows from mongodb


Loaded from: MONGODB
Shape: (100, 15)


,key,edition_key,title,overview,author_name,publisher,genres,subject_facet,language,first_publish_year,edition_count,number_of_pages_median,rating,ratings_count,has_fulltext
0,/works/OL893415W,OL28969075M,Dune,Unknown,Frank Herbert,Penguin Classics,Dune (imaginary place),"['American Science fiction', 'American literat...",gre,1965,120,592.0,4.301932,414.0,True
1,/works/OL893526W,OL29157219M,Dune Messiah,Unknown,Frank Herbert,Putnam Adult,American science fiction,"['American Science fiction', 'Anglais (langue)...",dut,1969,70,278.0,3.946970,132.0,True
2,/works/OL893516W,OL58638429M,Children Of Dune,Unknown,Frank Herbert,Berkley Books,Dune (imaginary place),"['American Science fiction', 'Dune (Imaginary ...",pol,1976,51,479.0,4.000000,42.0,True
3,/works/OL893502W,OL59974704M,Heretics Of Dune,Unknown,Frank Herbert,Berkley Books,Science fiction,"['American Science fiction', 'Anglais (langue)...",rus,1984,35,510.0,3.860000,50.0,True
4,/works/OL893515W,OL28285129M,God Emperor Of Dune,Unknown,Frank Herbert,Laffont,American science fiction,"['American Science fiction', 'Dune (Imaginary ...",fre,1981,40,464.0,3.945946,74.0,True


In [5]:
# Helpers that drive the UI controls
genres = da.get_genres(df)
year_min, year_max = da.get_year_bounds(df)
print(f"{len(genres)} distinct genres, e.g. {genres[:5]}")
print("Year range:", year_min, "to", year_max)

56 distinct genres, e.g. ['1990s', '2.', 'American science fiction', 'Art', 'Biography']
Year range: 1893 to 2023


In [6]:
# filter_books() is exactly what the callbacks call on every user interaction
subset = da.filter_books(genre=None, year_range=[2000, 2020], search="dune")
print("Books matching 'dune', 2000-2020:", len(subset))
subset[["title", "genres", "first_publish_year", "rating", "ratings_count"]].head()

Books matching 'dune', 2000-2020: 26


,title,genres,first_publish_year,rating,ratings_count
8,Dune,Science fiction,2001,3.266667,15.0
9,Sandworms Of Dune,Life on other planets,2007,3.411765,17.0
12,Navigators Of Dune,Fiction,2016,4.000000,7.0
21,The Road To Dune,Literary collections,2005,4.000000,3.0
22,Dune,"Fiction, science fiction, action & adventure",2004,5.000000,2.0


## Part 5 — Callback behavior (figure builders)

Each chart's logic lives in a pure `build_*_figure(df)` function in
`callbacks.py`; the `@app.callback` wrappers only read the filter values and
call these builders. That lets us render every chart right here in the
notebook — exactly what the dashboard shows when a user changes a filter.

In [7]:
from dashboard import callbacks as cb

# Filter as a user would, then build the four filter-driven charts
data = da.filter_books(year_range=[1950, 2023])

fig_bar = cb.build_top_books_figure(data)
fig_box = cb.build_rating_figure(data)
fig_scatter = cb.build_scatter_figure(data)
fig_trend = cb.build_trend_figure(data)

print("Charts built:",
      {"top_books": len(fig_bar.data),
       "rating_box": len(fig_box.data),
       "scatter": len(fig_scatter.data),
       "trend": len(fig_trend.data)})

Charts built: {'top_books': 1, 'rating_box': 10, 'scatter': 1, 'trend': 1}


In [8]:
fig_bar.show()    # Top books by reach (ratings count) — the bar-chart callback

In [9]:
fig_trend.show()  # Books published per year — an additional chart callback

In [10]:
# The live ticker is driven by dcc.Interval. Simulate a few ticks:
import random
from datetime import datetime
for _ in range(8):
    cb._LIVE_BUFFER.append((datetime.now().strftime("%H:%M:%S"), round(random.uniform(50, 500), 1)))
cb.build_live_figure().show()

## Part 6 — Run the dashboard locally

```bash
python app.py
# then open http://127.0.0.1:8050
```

You should see the dark dashboard with KPI cards, the genre/year/search
filters, four charts that all respond to the filters, and the live ticker
updating every two seconds. `app.py` also exposes `server` (the Flask object)
so Gunicorn can serve it in production: `gunicorn app:server`.

## Part 7 — Containerised deployment with Docker Compose

The `Dockerfile` builds a `python:3.12-slim` image that runs the app with
**Gunicorn**. `docker-compose.yml` defines two services — `web` (the
dashboard) and `db` (MongoDB) — on a shared network, so the app reaches the
database at `mongodb://db:27017/`. A named volume keeps Mongo data across
restarts.

## 📦 Deployment Documentation

### Prerequisites
- **Docker Desktop** installed and running (provides `docker` + `docker compose`).
- **Ports available** on the host: **8050** (dashboard) and **27017** (MongoDB).
  Close anything already using them.
- Python + project dependencies on the host *only* if you want to run the seed
  script from the host (`pip install pymongo pandas`).

### Start the stack and seed the database
```bash
# 1. Build the image and start both containers (dashboard + MongoDB)
docker compose up --build            # add -d to run detached

# 2. In a second terminal, seed MongoDB from the host
#    (the db service publishes 27017, so localhost works)
python scripts/seed_mongo.py
```

### Verify it's running
- Open **http://localhost:8050** in a browser.
- You should see the **Book Analytics — Live Dashboard**: four KPI cards, the
  genre dropdown / year range slider / title search box, four charts, and the
  **LIVE** ticker animating every ~2 seconds.
- Changing any filter should immediately redraw the four charts and KPI cards.
- The page header shows the data source (`MONGODB` once seeded, `CSV` otherwise).

### Stop the stack cleanly
```bash
docker compose down          # stop & remove the containers (keeps Mongo data)
docker compose down -v        # also remove the MongoDB volume (wipes data)
```

### Troubleshooting
- **Port 8050 already in use** — either stop the other process, or remap the
  port in `docker-compose.yml` (e.g. `"8060:8050"`) and open the new port.
  Find the culprit with `netstat -ano | findstr :8050` (Windows) /
  `lsof -i :8050` (macOS/Linux).
- **Dashboard shows the CSV source instead of MongoDB** — the `books`
  collection is empty; run `python scripts/seed_mongo.py` again.
- **`web` can't reach the database** — make sure you use the service name
  `db` (not `localhost`) inside the container; that's what the `MONGO_URI`
  env var in `docker-compose.yml` does.
- **Rebuild after code changes** — `docker compose up --build` forces a fresh
  image build.

## Assignment 13 — Solution summary

| Requirement | Where it's implemented |
|---|---|
| **Dash app** with `html`, `dcc`, `dash-bootstrap-components` | `app.py`, `src/dashboard/layout.py` |
| ≥ 3 interactive components | genre dropdown, year range slider, title search box (+ `dcc.Interval`) |
| ≥ 4 charts responding to the same filters | top-books bar, rating box-plot, editions-vs-reach scatter, books-per-year trend |
| ≥ 5 callbacks, each with ≥1 Input/Output | 6 callbacks in `callbacks.py` (4 charts + KPI cards + live ticker) |
| Live-updating ticker via `dcc.Interval` | `live-ticker` callback in `callbacks.py` |
| Bar-chart callback + one more chart callback | `build_top_books_figure` + `build_trend_figure` (and others) |
| Read from MongoDB, fall back to CSV | `src/dashboard/data_access.py` |
| Seed script populates the collection | `scripts/seed_mongo.py` |
| Dockerfile: `python:3.12-slim` + Gunicorn CMD | `Dockerfile` |
| `docker-compose.yml`: `web` + `db` services | `docker-compose.yml` |
| `.dockerignore` keeps the image lean | `.dockerignore` |
| Deployment documentation | the **📦 Deployment Documentation** section above |

**The 6 callbacks (all with ≥1 Input and 1 Output):**

| Pattern | Callback | Trigger | Output |
|---|---|---|---|
| N → 1 | Top-books bar | genre + year + search | `top-books-bar.figure` |
| N → 1 | Rating box-plot | genre + year + search | `rating-box.figure` |
| N → 1 | Editions-vs-reach scatter | genre + year + search | `scatter-plot.figure` |
| N → 1 | Books-per-year trend | genre + year + search | `trend-line.figure` |
| N → N | KPI cards | genre + year + search | four card values |
| 1 → 1 | Live ticker | `dcc.Interval` tick | `live-ticker.figure` |